# Lab05.1 – Traffic Sign Color Segmentation

**Objective:** extract red and blue road-sign regions using HSV.

Images are supplied in `images/`. No USB webcam or FPGA overlay is required.

**Step 1 — Preparation.** Import OpenCV, NumPy and Matplotlib; set the image directory and define a function that displays BGR images correctly as RGB. The OpenCV version is printed to verify the environment.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMAGE_DIR = Path('images')
def show_bgr(img, title='Image'):
    plt.figure(figsize=(11, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

print('OpenCV:', cv2.__version__)

**Step 2 — Input image.** Read `traffic_signs.png` from `images/` and display the six reference signs. Check that the image appears before continuing.

In [ ]:
img = cv2.imread(str(IMAGE_DIR / 'traffic_signs.png'))
assert img is not None, 'Image missing: upload the entire Lab05 directory'
show_bgr(img, 'Input: six traffic signs')

## 1. HSV masks
Red has two hue intervals because it wraps around zero in OpenCV HSV.

**Step 3 — HSV segmentation function.** Define the red and blue colour ranges in HSV. Red needs two hue intervals, and morphological opening removes small isolated regions. This cell only defines a function; it does not display anything.

In [ ]:
def color_masks(bgr):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    # OpenCV hue uses 0..179. Red wraps around zero.
    red1 = cv2.inRange(hsv, (0, 90, 70), (12, 255, 255))
    red2 = cv2.inRange(hsv, (168, 90, 70), (179, 255, 255))
    red_mask = cv2.bitwise_or(red1, red2)
    blue_mask = cv2.inRange(hsv, (95, 90, 60), (130, 255, 255))
    kernel = np.ones((3, 3), np.uint8)
    red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, kernel)
    blue_mask = cv2.morphologyEx(blue_mask, cv2.MORPH_OPEN, kernel)
    return {'red': red_mask, 'blue': blue_mask}

**Step 4 — Colour masks.** Execute segmentation and display the two binary masks. White denotes pixels selected by the colour filter.

In [ ]:
m = color_masks(img)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, mask) in zip(axes, m.items()):
    ax.imshow(mask, cmap='gray', vmin=0, vmax=255)
    ax.set_title(name.upper() + ' mask'); ax.axis('off')
plt.show()

**Step 5 — Segmented image.** Combine red and blue masks and show the pixels preserved by the filter. Compare the result with the original.

In [ ]:
combined = cv2.bitwise_or(m['red'], m['blue'])
segmented = cv2.bitwise_and(img, img, mask=combined)
show_bgr(segmented, 'Only red and blue pixels retained')

## Student tasks
1. Change the lower saturation limit from 90 to 140. Which regions disappear?
2. Explain why a single interval near hue 0 does not capture every shade of red.
3. Compare the number of nonzero pixels in the red and blue masks.

**Step 6 — Quantitative comparison.** Count the selected pixels for each colour and compare the counts after changing the HSV threshold.

In [ ]:
print('Red pixels:', cv2.countNonZero(m['red']))
print('Blue pixels:', cv2.countNonZero(m['blue']))